# Memory audit: where an ion_gym session's memory actually goes

Self-contained — nothing to import from outside this notebook. Run the
audit cell between flights and read the **deltas**.

### About the `app` argument

There is no app ID to pass. `SimApp._runs` is an attribute of a **Panel
app object**, which only exists inside `panel serve sim_app.py`. A Jupyter
kernel is a different process and cannot reach it, and there is no handle,
ID or port that would let it.

In a notebook **you are the run store**: every `model`, `fly`, `results`
you bind is held by your own namespace, plus IPython's `Out` cache. So the
audit below weighs *your namespace* instead, which is the notebook
equivalent of `app._runs`. (If you ever want the app's own numbers, run the
audit inside a code cell of the app itself, not from here.)

## Stage 0 — the audit

Read-only. Reports current RSS, every in-process ion_gym cache by name,
the biggest objects in your own namespace, IPython's output cache, and
whether a traceback is pinning frames.

`psutil` is worth installing: on macOS the stdlib has **no** current-RSS
source, only `ru_maxrss`, which is a peak and never falls. Without psutil
the number below is labelled `PEAK` so it can't be misread as current.

In [ ]:
import gc
import sys
import types

# Every module-level cache ion_gym keeps. Explicit, because a cache that
# is missing from this list is a GAP IN THE AUDIT, not proof of absence —
# so an absent attribute is REPORTED, never skipped.
CACHES = [
    ("ion_gym.physics.build_planar", "_PLANAR_BASIS_CACHE", "UNBOUNDED"),
    ("ion_gym.physics.build_rz",     "_RZ_BASIS_CACHE",     "UNBOUNDED"),
    ("ion_gym.physics.build_stl",    "_STL_BUILD_CACHE",    "UNBOUNDED"),
    ("ion_gym.physics.build_stl3d",  "_COMPOSE_GRAD_CACHE", "self-clearing"),
    ("ion_gym.viz.viz_core",         "_MASK_OUTLINE_CACHE", "LRU 256"),
]


def rss():
    """(bytes, label). psutil -> /proc -> getrusage(PEAK, labelled)."""
    try:
        import psutil
        return psutil.Process().memory_info().rss, "current"
    except ImportError:
        pass
    try:
        with open("/proc/self/status") as fh:
            for line in fh:
                if line.startswith("VmRSS:"):
                    return int(line.split()[1]) * 1024, "current"
    except FileNotFoundError:
        pass
    import resource
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    r = r if sys.platform == "darwin" else r * 1024
    return r, "PEAK — install psutil for current rss"


SKIPPED = []          # (type, error) for anything the walk could not weigh


def nbytes_of(obj, seen=None, depth=0):
    """Array bytes reachable from obj. Bounded depth so a cyclic object
    graph cannot hang the audit."""
    if seen is None:
        seen = set()
    if depth > 6 or id(obj) in seen:
        return 0
    seen.add(id(obj))
    n = getattr(obj, "nbytes", None)
    if isinstance(n, int):
        return n
    t = 0
    if isinstance(obj, dict):
        for k, v in list(obj.items()):
            t += nbytes_of(k, seen, depth + 1) + nbytes_of(v, seen, depth + 1)
    elif isinstance(obj, (list, tuple, set, frozenset)):
        for v in list(obj):
            t += nbytes_of(v, seen, depth + 1)
    elif isinstance(obj, types.ModuleType):
        # Modules are never data, and walking one can trigger a LAZY
        # IMPORT of an optional backend that is not installed (measured:
        # touching a module attribute pulled in `dbm`, which raised
        # ImportError for _gdbm and killed the audit). Skipped by kind,
        # deliberately, not caught after the fact.
        return 0
    elif hasattr(obj, "__dict__"):
        try:
            t += nbytes_of(vars(obj), seen, depth + 1)
        except Exception as e:                      # noqa: BLE001
            # A skipped item is a REPORTED item: record it so the total
            # is never quietly short, and carry on with the rest.
            SKIPPED.append((type(obj).__name__, f"{type(e).__name__}: {e}"))
    return t


def audit(top=8, collect=True):
    mb = lambda b: f"{b / 1e6:10.1f} MB"          # noqa: E731
    r, lbl = rss()
    print("=" * 64)
    print(f"RSS {mb(r)}  [{lbl}]")
    print("-" * 64)

    total = 0
    for mod, attr, note in CACHES:
        m = sys.modules.get(mod)
        if m is None:
            print(f"  {attr:22s}  module not imported")
            continue
        c = getattr(m, attr, None)
        if c is None:
            print(f"  {attr:22s}  ABSENT — audit is out of date for this "
                  f"build; check {mod}")
            continue
        b = nbytes_of(c)
        total += b
        print(f"  {attr:22s} {len(c):>4} entries {mb(b)}  ({note})")
    print(f"  {'ion_gym caches total':22s} {'':>4}         {mb(total)}")

    ns = get_ipython().user_ns
    rows = []
    for k, v in list(ns.items()):
        if k.startswith("_") or k in ("In", "Out", "exit", "quit"):
            continue
        if isinstance(v, types.ModuleType):
            continue
        b = nbytes_of(v)
        if b > 1e6:
            rows.append((b, k, type(v).__name__))
    rows.sort(reverse=True)
    print("-" * 64)
    print(f"  your namespace: {len(rows)} object(s) over 1 MB")
    for b, k, tn in rows[:top]:
        print(f"    {k:28s} {tn:18s} {mb(b)}")
    ns_total = sum(b for b, _, _ in rows)
    print(f"    {'namespace total':28s} {'':18s} {mb(ns_total)}")

    out = ns.get("Out", {})
    ob = nbytes_of(out)
    print("-" * 64)
    print(f"  IPython Out cache: {len(out)} entries {mb(ob)}")
    if ob > 1e8:
        print("    ** a cell ended in a bare expression holding arrays. **")
        print("    ** `%reset -f out` drops it; see Stage 2.             **")
    tb = getattr(sys, "last_traceback", None)
    print(f"  sys.last_traceback: "
          + ("none" if tb is None else
             "PRESENT — pinning every local of every frame, "
             "including solver arrays. See Stage 2."))

    if SKIPPED:
        print("-" * 64)
        print(f"  {len(SKIPPED)} object(s) could not be weighed "
              f"(total above is a LOWER BOUND):")
        for tn, err in SKIPPED[:5]:
            print(f"    {tn:28s} {err[:40]}")
        SKIPPED.clear()

    if collect:
        n = gc.collect()
        r2, lbl2 = rss()
        print("-" * 64)
        print(f"  gc.collect(): {n} object(s), "
              f"{len(gc.garbage)} uncollectable")
        print(f"  RSS after     {mb(r2)}  [{lbl2}]")
        print("  NOTE: rss can stay flat after a REAL free — the allocator")
        print("        keeps the arena. Trust the byte counts above.")
    print("=" * 64)
    return dict(rss=r, caches=total, namespace=ns_total, out=ob)


baseline = audit()

## Stage 1 — clear every cache

All three unbounded ones, plus the self-clearing gradient cache. `build_stl`
has **no** `clear_memory_cache()` — its dict is cleared directly here, which
is a gap in the library, not a preference.

In [ ]:
def clear_all_caches(verbose=True):
    """Drop every in-process cache. Returns {name: entries_dropped}.

    Safe: the disk basis cache (fa_cache) still backs all of it, so the
    next build re-loads rather than re-solves.
    """
    dropped = {}
    from ion_gym.physics.build_planar import clear_memory_cache as cp
    from ion_gym.physics.build_rz import clear_memory_cache as cr
    dropped["planar"] = cp()
    dropped["rz"] = cr()
    # no clear_memory_cache() exists for these two
    from ion_gym.physics import build_stl, build_stl3d
    dropped["stl"] = len(build_stl._STL_BUILD_CACHE)
    build_stl._STL_BUILD_CACHE.clear()
    dropped["compose_grad"] = len(build_stl3d._COMPOSE_GRAD_CACHE)
    build_stl3d._COMPOSE_GRAD_CACHE.clear()
    gc.collect()
    if verbose:
        print("dropped: " + ", ".join(f"{k}={v}" for k, v in dropped.items()))
    return dropped


clear_all_caches()
audit();

## Stage 2 — release what the notebook itself is holding

`del model` is **not** enough: IPython's `Out[n]` and `_7` still reference
it. `%xdel` clears those too. A traceback from a failed cell pins every
local of every frame — including a multi-GB basis array — until cleared.

In [ ]:
# Drop IPython's output cache — every cell that ended in a bare
# expression. displayhook.flush() is exactly what `%reset out` calls,
# without the magic: magics route through the history manager, which
# needs gdbm and is absent on some installs.
get_ipython().displayhook.flush()

# release a traceback still pinning solver frames
sys.last_traceback = sys.last_value = sys.last_type = None

# close retained figures (matplotlib's registry holds them, not your var)
try:
    import matplotlib.pyplot as plt
    plt.close("all")
except ImportError:
    print("matplotlib not imported — nothing to close")

gc.collect()
audit();

## Stage 3 — a sweep that stays flat

Point `DECK` at your deck and `PITCHES` at what you want to compare. The
shape that matters: **keep scalars, drop arrays, clear between variants.**

`del` plus `displayhook.flush()` releases a binding fully; `del` alone leaves it alive in `Out[n]`.

In [ ]:
from pathlib import Path

from ion_gym.io.paths import repo_root
from ion_gym.io.sim_spec import SimSpec, set_pitch
from ion_gym.physics.sim_build import build_run, sizing_for

# ---- named parameters -------------------------------------------------
DECK = Path(repo_root()) / "internal/studies/bent_flatapole/decks" \
                           "/bent_flatapole_channel.json"
PITCHES = [0.4, 0.3, 0.2]      # mm/gu, coarse -> fine
GB_BUDGET = 8.0                # refuse a variant whose bases exceed this

summary = []
for pitch in PITCHES:
    spec = SimSpec.from_json(str(DECK))
    set_pitch(spec, pitch)                 # snaps the domain; never bare mm_per_gu
    z = sizing_for(spec)
    gb = z.mem_bases_bytes / 1e9
    if gb > GB_BUDGET:
        # REFUSE with the number, rather than starting a solve that will
        # take the kernel down and lose the variants already finished
        print(f"SKIP pitch {pitch}: {gb:.2f} GB of bases exceeds the "
              f"{GB_BUDGET} GB budget ({z.dims}, {z.n_electrodes} bases)")
        summary.append(dict(pitch=pitch, skipped=True, gb=gb))
        continue
    print(f"pitch {pitch}: {z.dims}, {z.n_electrodes} bases, {gb:.2f} GB, "
          f"est {z.est_solve_s / 60:.1f} min")
    model, fly, cols, births = build_run(spec)
    summary.append(dict(pitch=pitch, skipped=False, gb=gb,
                        dims=z.dims, n_bases=z.n_electrodes))
    del model, fly, cols, births, spec
    get_ipython().displayhook.flush()   # Out[] still holds them
    clear_all_caches(verbose=False)
    audit(top=3)

print(summary)

## Read-out

- **The audit's byte counts are the evidence, not RSS.** A real free often
  leaves RSS flat because the allocator keeps the arena; on macOS without
  `psutil` the number is a peak and cannot fall at all.
- **The unbounded caches are keyed by geometry AND pitch**, so a sweep adds
  a full basis set per variant and never evicts. That is the growth, and
  `clear_all_caches()` between variants is what flattens it.
- **The notebook's own retention is separate and often larger**: `Out`, a
  pinned traceback, and matplotlib's figure registry all hold arrays your
  variables no longer name.
- **Refusing a variant beats killing the kernel.** `sizing_for` gives the
  basis bytes before any solve starts, which is why the sweep checks a
  budget instead of finding out at 50 GB.